# 第 14 章 · 强化学习 GRPO / PPO / CISPO

> 📄 配套动画网页：`../14_rl.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

DPO 只能被动学现成数据。强化学习让模型自己生成、自己评分、自己改进。MiniMind 把四种算法统一成一个公式。

---


## 🎯 本章目标

- 理解「策略项 × 优势项 × 正则项」的统一视角
- 掌握 `torch.distributions.Categorical` 采样与 log_prob
- 掌握 `torch.clamp` 实现 PPO 裁剪
- 能实现 GRPO 的优势归一化与策略损失

---


## 📖 背景：统一视角

```
J = E[ Φ(r_t, A_t) ]  −  β · KL(π ‖ π_ref)
      └─ 策略项 ─┘        └─ 正则项 ─┘
                ↑
          优势项 A_t：这一步比平均好多少
```

| 算法 | 策略项 Φ | 优势 A_t | 模型数 |
|---|---|---|---|
| DPO | log r_w − log r_l | 隐式 | 1（2 次前向） |
| PPO | min(r·A, clip(r)·A) | Critic / GAE | 2 |
| GRPO | min(r·A, clip(r)·A) | (R−μ)/σ | 1 |
| CISPO | clip(r)·A·logπ | (R−μ)/σ | 1 |

**GRPO 最巧妙的地方**：不需要 Critic。对同一个问题采样一组回答，
用组内奖励的均值/标准差做归一化，就得到了优势。

---



## 🖼 官方配图

![MiniMind GRPO 训练曲线：奖励稳定上升](../images/grpo_loss.jpg)

*MiniMind GRPO 训练曲线：奖励稳定上升*

![MiniMind PPO 训练曲线：需要 actor + critic 双网络协同](../images/ppo_loss.jpg)

*MiniMind PPO 训练曲线：需要 actor + critic 双网络协同*

---


# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 14.1 采样：`torch.distributions.Categorical`

In [ ]:
import torch
from torch.distributions import Categorical

logits = torch.tensor([[2.0, 1.0, 0.1, 0.1]])
dist = Categorical(logits=logits)
samples = dist.sample((5,))               # 采样 5 次
print("采样结果:", samples.tolist())
print("对应 log 概率:", dist.log_prob(samples).tolist())
print("概率分布:", dist.probs.tolist())

### 14.2 `log_prob` 与 `gather` 取特定动作

In [ ]:
import torch.nn.functional as F

logits = torch.randn(2, 4)                    # [B, num_actions]
actions = torch.tensor([1, 3])                # 实际采取的动作

logp = F.log_softmax(logits, dim=-1)
picked = logp.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
print("动作的 log 概率:", picked.tolist())

### 14.3 组内归一化 → 优势

In [ ]:
rewards = torch.tensor([0.9, 0.2, 0.6, 0.1, 0.8, 0.3, 0.5, 0.4])
mu, sd = rewards.mean(), rewards.std()
advantages = (rewards - mu) / (sd + 1e-8)
print(f"μ = {mu:.3f}   σ = {sd:.3f}")
print("优势:", [round(a, 3) for a in advantages.tolist()])
print("优势均值≈0:", round(advantages.mean().item(), 6), " 标准差≈1:", round(advantages.std().item(), 4))

### 14.4 PPO 裁剪：`torch.clamp`

In [ ]:
eps = 0.2
ratio = torch.tensor([0.5, 0.9, 1.0, 1.1, 1.5, 2.0])
A = 1.0                                    # 正优势

clipped = torch.clamp(ratio, 1 - eps, 1 + eps)
obj = torch.minimum(ratio * A, clipped * A)
print("ratio  :", ratio.tolist())
print("clipped:", clipped.tolist())
print("目标   :", obj.tolist(), " <- ratio 超过 1.2 后不再增长")

### 14.5 `detach` 的重要性

计算 `r = π_new / π_old` 时，分母必须 `detach`，否则梯度会走错方向。

In [ ]:
logp_new = torch.tensor([-1.0], requires_grad=True)
logp_old = torch.tensor([-1.2])

r_wrong = torch.exp(logp_new - logp_old)              # 分母带梯度
r_right = torch.exp(logp_new - logp_old.detach())     # 分母 detach

print("两种写法数值相同:", torch.allclose(r_wrong, r_right))
print("但对 logp_old 的梯度不同（正确写法里 logp_old 不应有梯度）")

### 14.6 KL 正则：用 log 概率近似

In [ ]:
logp = torch.tensor([[-1.0, -2.0, -3.0]])
logp_ref = torch.tensor([[-1.2, -1.8, -3.1]])

# 无偏估计：exp(logp_ref - logp) - (logp_ref - logp) - 1
diff = logp_ref - logp
kl = torch.exp(diff) - diff - 1
print("近似 KL:", kl.tolist(), " <- 每项都 >= 0")

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

现在实现 GRPO 的核心部分。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 14.7 实现 `compute_advantage`

**要求**：输入一组奖励 `[G]`，返回归一化后的优势 `[G]`。

```python
A = (R - R.mean()) / (R.std() + 1e-8)
```

In [ ]:
# ======== 你的实现：compute_advantage ========
#
# 【要求】
#   - 签名：compute_advantage(rewards)
#   - rewards: 形状 [G] 的张量（G = 组内采样数）
#   - 返回 (rewards - rewards.mean()) / (rewards.std() + 1e-8)
#
# 【可用到的函数 / 方法】
#   - `Tensor.mean`、`Tensor.std`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：compute_advantage ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    r = torch.tensor([0.9, 0.2, 0.6, 0.1, 0.8, 0.3, 0.5, 0.4])
    a = compute_advantage(r)
    assert tuple(a.shape) == tuple(r.shape), f"优势应与奖励同形状，得到 {tuple(a.shape)}"
    close(a.mean().item(), 0.0, tol=1e-5, msg="优势的均值应约为 0")
    assert abs(a.std().item() - 1.0) < 0.1, f"优势的标准差应约为 1，得到 {a.std().item():.3f}"
    assert a.max() > 0 and a.min() < 0, "奖励高于/低于平均的样本应分别得到正/负优势"

check("compute_advantage 组内归一化", _t)

### 14.8 实现 `grpo_loss`

**要求**：
1. 计算概率比 `ratio = exp(logp_new - logp_old.detach())`
2. 计算 `surr1 = ratio * A`，`surr2 = clamp(ratio, 1-ε, 1+ε) * A`
3. 策略损失 `policy_loss = -min(surr1, surr2).mean()`
4. 加上 KL 正则项（可选，用上面的近似公式）
5. 返回总损失

**注意**：`logp_old` 一定要 `detach()`。

In [ ]:
# ======== 你的实现：grpo_loss ========
#
# 【要求】
#   - 签名：grpo_loss(logp_new, logp_old, advantages, eps=0.2, kl_coef=0.0, logp_ref=None)
#   - ratio = torch.exp(logp_new - logp_old.detach())
#   - surr1 = ratio * advantages；surr2 = clamp(ratio, 1-eps, 1+eps) * advantages
#   - policy_loss = -torch.min(surr1, surr2).mean()
#   - 若有 logp_ref，加上 kl_coef * KL（用 exp(diff) - diff - 1 近似）
#   - 返回总损失
#
# 【可用到的函数 / 方法】
#   - `torch.exp`、`torch.clamp`、`torch.minimum`/`torch.min`
#   - `Tensor.detach`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：grpo_loss ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    logp_new = torch.randn(8, requires_grad=True)
    logp_old = torch.randn(8, requires_grad=True)
    adv = torch.randn(8)

    out = grpo_loss(logp_new, logp_old, adv)
    loss = out[0] if isinstance(out, tuple) else out
    assert torch.isfinite(loss) and loss.dim() == 0, f"应返回有限标量，得到 {loss}"
    loss.backward()
    assert logp_new.grad is not None and torch.isfinite(logp_new.grad).all(), "梯度应能传回 logp_new"
    assert logp_old.grad is None, "logp_old 不该有梯度 —— 计算 ratio 时忘记 .detach() 了"

    z = torch.zeros(8, requires_grad=True)
    out2 = grpo_loss(z, torch.zeros(8), adv)
    loss2 = out2[0] if isinstance(out2, tuple) else out2
    close(loss2.item(), (-adv.mean()).item(), tol=1e-5,
          msg="ratio=1 时 min(r·A, clip(r)·A) 应等于 A，loss 应为 -mean(A)")

check("grpo_loss 数值 / ratio=1 特例 / detach 与反传", _t)

### 14.9 实现 `cispo_loss`（进阶）

CISPO 与 GRPO 的差别只在策略项：它**不裁剪 ratio 本身，而是把 ratio 当作权重**，
再乘上 `log π`。

```python
ratio_clipped = clamp(ratio, 0, eps_high)      # 注意下界是 0
policy_loss = -(ratio_clipped * advantages * logp_new).mean()
```

**思考**：这样在 ratio 被裁剪时，梯度依然能通过 `logp_new` 流动，这是 CISPO 的设计意图。

In [ ]:
# ======== 你的实现：cispo_loss ========
#
# 【要求】
#   - 签名：cispo_loss(logp_new, logp_old, advantages, eps=0.2)
#   - ratio = torch.exp(logp_new - logp_old.detach())
#   - ratio_clipped = torch.clamp(ratio, 0, 1 + eps)
#   - policy_loss = -(ratio_clipped * advantages * logp_new).mean()
#   - 返回 policy_loss
#
# 【可用到的函数 / 方法】
#   - `torch.clamp`、`Tensor.mean`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：cispo_loss ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    logp_new = torch.randn(8, requires_grad=True)
    logp_old = torch.randn(8)
    adv = torch.randn(8)

    out = cispo_loss(logp_new, logp_old, adv, eps=0.2)
    loss = out[0] if isinstance(out, tuple) else out
    assert torch.isfinite(loss) and loss.dim() == 0, f"应返回有限标量，得到 {loss}"
    loss.backward()
    assert logp_new.grad is not None and torch.isfinite(logp_new.grad).all(), "梯度应能传回 logp_new"

    big = torch.full((4,), 5.0, requires_grad=True)      # ratio = e^5，远超 1+eps
    out2 = cispo_loss(big, torch.zeros(4), torch.ones(4), eps=0.2)
    loss2 = out2[0] if isinstance(out2, tuple) else out2
    loss2.backward()
    assert big.grad.abs().sum() > 0, \
        "ratio 被裁剪后梯度消失了 —— CISPO 的设计意图正是让梯度继续通过 logp_new 流动"

check("cispo_loss 数值 / 裁剪后仍有梯度", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能用统一公式说出四种算法的区别
- [ ] GRPO 为什么不需要 Critic？
- [ ] PPO 的 clip 在 ratio 超过上界后做了什么？
- [ ] `logp_old.detach()` 忘了写会怎样？
- [ ] 优势归一化后均值≈0、标准差≈1，验证了吗？

**思考题**：如果一组回答的奖励全都相同（σ=0），优势会变成什么？这时 GRPO 还有梯度吗？
